# VIIRS Batch-Style Playground

Use this notebook as a compact batch-ready example for VIIRS processing with SpiPy.

Workflow:
- configure a VIIRS product family, LUT, summer scenes, and inversion-scene glob
- build or reuse a saved `R0` under `data/viirs/r0/<sensor>/<tile>/<year>/`
- loop over matching inversion scenes
- compute each inversion, save a 1x4 quicklook figure, and write a netCDF result
- inspect the log tail


In [ ]:
from pathlib import Path
import sys
from time import perf_counter

from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import xarray as xr

search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(path for path in search_roots if (path / 'spires').exists() and (path / 'tests' / 'data').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from spires import configure_spires_file_logger, make_spires_log_path
from spires.sensors import build_r0_from_sources, prepare_scene_for_inversion, run_inversion
from spires.sensors.viirs import parse_viirs_surface_reflectance_filename

SENSOR_LABELS = {
    'snpp': 'SNPP',
    'noaa20': 'NOAA20',
    'noaa21': 'NOAA21',
}
LUT_BY_SENSOR = {
    'snpp': repo_root / 'data' / 'viirs' / 'lut' / 'lut_viirs_snpp_i1_i2_i3_m2_m4_m8_m11_3um_dust_bandpass.mat',
    'noaa20': repo_root / 'data' / 'viirs' / 'lut' / 'lut_viirs_noaa20_i1_i2_i3_m2_m4_m8_m11_3um_dust_bandpass.mat',
    'noaa21': repo_root / 'data' / 'viirs' / 'lut' / 'lut_viirs_noaa21_i1_i2_i3_m2_m4_m8_m11_3um_dust_bandpass.mat',
}

# Adjust this to choose which sensor-family input directory to process.
# Input HDFs are expected under data/viirs/inputs/<sensor>/r0/ and /reflectance/.
sensor = 'noaa21'
input_root = repo_root / 'data' / 'viirs' / 'inputs' / sensor
r0_input_dir = input_root / 'r0'
reflectance_input_dir = input_root / 'reflectance'
r0_input_glob = '**/*.h5'
inversion_glob = '*.h5'
cloud_mask_policy = 'ignore_cloud_and_shadow'
ZARR_CHUNKS = {'time': 1, 'y': 512, 'x': 512, 'band': -1}

r0_source_paths = sorted(r0_input_dir.glob(r0_input_glob))
scenes_for_inversion = sorted(reflectance_input_dir.glob(inversion_glob))

# Backward-compatible aliases used by the cells below.
summer_paths = r0_source_paths
inversion_paths = scenes_for_inversion

reference_path = scenes_for_inversion[0] if scenes_for_inversion else (r0_source_paths[0] if r0_source_paths else None)
reference_meta = parse_viirs_surface_reflectance_filename(reference_path) if reference_path is not None else None
if reference_meta is not None:
    sensor = reference_meta.platform
tile = reference_meta.tile if reference_meta is not None else 'h08v05'
lut_path = LUT_BY_SENSOR[sensor]

r0_meta = parse_viirs_surface_reflectance_filename(r0_source_paths[0]) if r0_source_paths else reference_meta
r0_year = int(r0_meta.acquisition_date[:4]) if r0_meta is not None else 2023
r0_dir = repo_root / 'data' / 'viirs' / 'r0' / sensor / tile / str(r0_year)
r0_dir.mkdir(parents=True, exist_ok=True)
r0_path = r0_dir / f'viirs_r0_{sensor}_{tile}_{r0_year}.nc'

output_root = repo_root / 'outputs' / 'viirs' / sensor
data_output_dir = output_root / 'data'
fig_output_dir = output_root / 'figs'
timeseries_dir = output_root / 'timeseries'
data_output_dir.mkdir(parents=True, exist_ok=True)
fig_output_dir.mkdir(parents=True, exist_ok=True)
timeseries_dir.mkdir(parents=True, exist_ok=True)
r0_zarr_path = timeseries_dir / f'viirs_{sensor}_{tile}_{r0_year}_r0_stack.zarr'

log_path = make_spires_log_path(repo_root / 'outputs', prefix='viirs_batch_playground', sensor=sensor, tile=tile)
logger = configure_spires_file_logger(log_path, logger_name='spires.examples.viirs', log_to_stdout=False)

run_summary_lines = [
    f'Repo root: {repo_root}',
    f'Input root: {input_root}',
    f'R0 input dir: {r0_input_dir}',
    f'R0 input glob: {r0_input_glob}',
    f'R0 source scenes found: {len(r0_source_paths)}',
    f'Reflectance input dir: {reflectance_input_dir}',
    f'Inversion glob: {inversion_glob}',
    f'Scenes for inversion found: {len(scenes_for_inversion)}',
    f'Sensor: {sensor}',
    f'Tile: {tile}',
    f'LUT: {lut_path}',
    f'R0 output: {r0_path}',
    f'Inversion data output dir: {data_output_dir}',
    f'Figure output dir: {fig_output_dir}',
    f'Timeseries zarr: {r0_zarr_path}',
    f'Log file: {log_path}',
]

for line in run_summary_lines:
    logger.info(line)
    print(line)


## Input Check

This notebook assumes that `r0_source_paths` and `scenes_for_inversion` are non-empty and that the scenes match the configured LUT platform and tile. If the next cell fails, update `sensor`, `r0_input_glob`, or `inversion_glob` first.


In [ ]:
assert r0_source_paths, f'No R0 source scenes matched {r0_input_glob!r} in {r0_input_dir}'
assert scenes_for_inversion, f'No inversion scenes matched {inversion_glob!r} in {reflectance_input_dir}'
assert lut_path.exists(), f'Missing VIIRS LUT: {lut_path}'

summer_meta = [parse_viirs_surface_reflectance_filename(path) for path in r0_source_paths]
inversion_meta = [parse_viirs_surface_reflectance_filename(path) for path in scenes_for_inversion]
all_meta = summer_meta + inversion_meta
assert {meta.platform for meta in all_meta} == {sensor}, 'Summer and inversion scenes must use one VIIRS platform.'
assert {meta.tile for meta in all_meta} == {tile}, 'Summer and inversion scenes must use one VIIRS tile.'

print('First three R0 source scenes:')
for path in r0_source_paths[:3]:
    print(' ', path.name)
print('First three scenes for inversion:')
for path in scenes_for_inversion[:3]:
    print(' ', path.name)


## Batch-Style Processing

This cell builds or reuses the `R0` file, then processes every scene matched by `inversion_glob`. For each scene it prepares the inputs, creates and computes the lazy inversion dataset, saves a 1x4 quicklook figure, and writes a sensor/date-specific netCDF.


In [ ]:
def plot_viirs_inversion_1x4(inversion_ds, *, sensor, acquisition_date, figure_path):
    sensor_label = SENSOR_LABELS.get(sensor, sensor.upper())
    fig, ax = plt.subplots(1, 4, figsize=(24, 5))
    (inversion_ds.raw_viewable_snow_fraction * 100).plot(ax=ax[0], cmap='YlGnBu_r', vmin=0, vmax=100)
    (inversion_ds.raw_shade_fraction * 100).plot(ax=ax[1], cmap='Greys', vmin=0, vmax=100)
    (inversion_ds.raw_canopy_fraction * 100).plot(ax=ax[2], cmap='YlGn', vmin=0, vmax=100)
    (inversion_ds.raw_snow_fraction * 100).plot(ax=ax[3], cmap='YlGnBu_r', vmin=0, vmax=100)

    ax[0].set_title('SpiPy raw viewable snow fraction')
    ax[1].set_title('SpiPy raw shade fraction')
    ax[2].set_title('SpiPy raw canopy fraction')
    ax[3].set_title('SpiPy raw snow fraction')
    fig.suptitle(f'VIIRS {sensor_label} // {acquisition_date}', y=1.02)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig

start_all = perf_counter()
r0_ds = build_r0_from_sources(
    r0_source_paths,
    sensor='viirs',
    platform=sensor,
    r0_path=r0_path,
    lut_file=lut_path,
    logger=logger,
    show_progress=True,
    cloud_mask_policy=cloud_mask_policy,
    zarr_path=r0_zarr_path,
    chunks=ZARR_CHUNKS,
)
print(f'R0 ready: {r0_path}')

processed_outputs = []
for scene_path in scenes_for_inversion:
    meta = parse_viirs_surface_reflectance_filename(scene_path)
    sensor_label = SENSOR_LABELS.get(meta.platform, meta.platform.upper()).lower()
    date_slug = meta.acquisition_date.replace('-', '')
    output_stem = f'viirs_{sensor_label}_{meta.tile}_{date_slug}'
    nc_path = data_output_dir / f'{output_stem}_inversion.nc'
    fig_path = fig_output_dir / f'{output_stem}_quicklook.png'

    print(f'Processing {scene_path.name}')
    scene_ds = prepare_scene_for_inversion(
        scene_path,
        sensor='viirs',
        platform=meta.platform,
        lut_file=lut_path,
        logger=logger,
        cloud_mask_policy=cloud_mask_policy,
    )
    inversion_ds = run_inversion(
        scene_ds,
        r0_ds,
        sensor='viirs',
        platform=meta.platform,
        lut_file=lut_path,
        logger=logger,
        execution_profile='local',
        apply_valid_inversion_mask=False,
    )

    start_scene = perf_counter()
    with ProgressBar(dt=5.0):
        inversion_ds = inversion_ds.compute()
    elapsed_scene = perf_counter() - start_scene
    print(f'Computed {scene_path.name} in {elapsed_scene / 60:.1f} minutes')

    plot_viirs_inversion_1x4(
        inversion_ds,
        sensor=meta.platform,
        acquisition_date=meta.acquisition_date,
        figure_path=fig_path,
    )
    inversion_ds.to_netcdf(nc_path)
    processed_outputs.append({'scene': scene_path, 'netcdf': nc_path, 'figure': fig_path})
    print(f'Saved netCDF: {nc_path}')
    print(f'Saved figure: {fig_path}')

elapsed_all = perf_counter() - start_all
print(f'Processed {len(processed_outputs)} scene(s) in {elapsed_all / 60:.1f} minutes')


## Saved Outputs and Log Tail

This confirms the saved R0 location, batch outputs, and recent workflow log events.


In [ ]:
saved_r0 = xr.open_dataset(r0_path)

print(saved_r0)
print()
print('Processed outputs:')
for item in processed_outputs:
    print(' ', item['scene'].name)
    print('    netCDF:', item['netcdf'])
    print('    figure:', item['figure'])
print()
print('Recent log lines:')
for line in log_path.read_text().splitlines()[-14:]:
    print(line)


## Grouping Benchmark

This benchmark reuses the existing NOAA-20 LUT, saved `R0`, and local reflectance scenes under `data/viirs/` to compare ungrouped inversion against the new `first` and `chunk_bin_mean` grouping modes.


In [ ]:
import gc
import numpy as np
import pandas as pd

benchmark_sensor = 'noaa20'
benchmark_tile = 'h08v05'
benchmark_lut_path = LUT_BY_SENSOR[benchmark_sensor]
benchmark_r0_path = (
    repo_root
    / 'data'
    / 'viirs'
    / 'r0'
    / benchmark_sensor
    / benchmark_tile
    / '2022'
    / f'viirs_r0_{benchmark_sensor}_{benchmark_tile}_2022.nc'
)
benchmark_scene_paths = sorted((repo_root / 'data' / 'viirs' / 'inputs' / benchmark_sensor / 'reflectance').glob('*.h5'))

assert benchmark_lut_path.exists(), f'Missing NOAA20 LUT: {benchmark_lut_path}'
assert benchmark_r0_path.exists(), f'Missing NOAA20 R0 file: {benchmark_r0_path}'
assert benchmark_scene_paths, 'No NOAA20 reflectance scenes found for benchmarking.'

benchmark_output_dir = repo_root / 'outputs' / 'viirs' / benchmark_sensor / 'benchmark'
benchmark_output_dir.mkdir(parents=True, exist_ok=True)

benchmark_methods = [
    {
        'label': 'ungrouped',
        'kwargs': {
            'use_grouping': False,
        },
    },
    {
        'label': 'first',
        'kwargs': {
            'use_grouping': True,
            'grouping_method': 'first',
            'grouping_tolerance': 0.05,
        },
    },
    {
        'label': 'chunk_bin_mean',
        'kwargs': {
            'use_grouping': True,
            'grouping_method': 'chunk_bin_mean',
            'grouping_tolerance': 0.05,
        },
    },
]

benchmark_results = []
benchmark_delta_arrays = {}
for scene_path in benchmark_scene_paths[:1]:
    meta = parse_viirs_surface_reflectance_filename(scene_path)
    print(f'\nBenchmark scene: {scene_path.name}')
    prepared_scene = prepare_viirs_scene_for_inversion(
        scene_path,
        lut_file=benchmark_lut_path,
        logger=logger,
        cloud_mask_policy=cloud_mask_policy,
    )

    per_method = {}
    for method in benchmark_methods:
        label = method['label']
        print(f'  Running {label}...')
        inversion_ds = run_viirs_inversion(
            prepared_scene,
            benchmark_r0_path,
            lut_file=benchmark_lut_path,
            logger=logger,
            execution_profile='local',
            apply_valid_inversion_mask=False,
            **method['kwargs'],
        )
        start = perf_counter()
        with ProgressBar(dt=5.0):
            computed = inversion_ds.compute()
        elapsed = perf_counter() - start
        per_method[label] = computed
        benchmark_results.append(
            {
                'scene': scene_path.name,
                'method': label,
                'seconds': elapsed,
                'minutes': elapsed / 60.0,
                'grouping_enabled': bool(computed.attrs.get('grouping_enabled', False)),
                'grouping_method': computed.attrs.get('grouping_method', 'none'),
            }
        )
        print(f'    {label}: {elapsed:.2f} s')
        del inversion_ds
        gc.collect()

    baseline = per_method['ungrouped']
    for label in ('first', 'chunk_bin_mean'):
        candidate = per_method[label]

        delta_viewable = candidate['raw_viewable_snow_fraction'] - baseline['raw_viewable_snow_fraction']
        delta_viewable_values = delta_viewable.values
        finite_viewable = np.isfinite(delta_viewable_values)
        flat_delta_viewable = delta_viewable_values[finite_viewable].ravel()
        benchmark_delta_arrays[(scene_path.name, label, 'raw_viewable_snow_fraction')] = flat_delta_viewable
        max_abs_delta_viewable = float(np.nanmax(np.abs(delta_viewable_values))) if np.any(finite_viewable) else float('nan')
        mean_abs_delta_viewable = float(np.nanmean(np.abs(delta_viewable_values))) if np.any(finite_viewable) else float('nan')

        delta_raw_snow = candidate['raw_snow_fraction'] - baseline['raw_snow_fraction']
        delta_raw_snow_values = delta_raw_snow.values
        finite_raw_snow = np.isfinite(delta_raw_snow_values)
        flat_delta_raw_snow = delta_raw_snow_values[finite_raw_snow].ravel()
        benchmark_delta_arrays[(scene_path.name, label, 'raw_snow_fraction')] = flat_delta_raw_snow
        max_abs_delta_raw_snow = float(np.nanmax(np.abs(delta_raw_snow_values))) if np.any(finite_raw_snow) else float('nan')
        mean_abs_delta_raw_snow = float(np.nanmean(np.abs(delta_raw_snow_values))) if np.any(finite_raw_snow) else float('nan')

        benchmark_results.append(
            {
                'scene': scene_path.name,
                'method': f'{label}_delta_vs_ungrouped',
                'seconds': np.nan,
                'minutes': np.nan,
                'grouping_enabled': True,
                'grouping_method': label,
                'max_abs_raw_viewable_snow_fraction_delta': max_abs_delta_viewable,
                'mean_abs_raw_viewable_snow_fraction_delta': mean_abs_delta_viewable,
                'max_abs_raw_snow_fraction_delta': max_abs_delta_raw_snow,
                'mean_abs_raw_snow_fraction_delta': mean_abs_delta_raw_snow,
            }
        )
        print(
            f'    {label} delta vs ungrouped: '
            f'max |Δviewable_snow_fraction|={max_abs_delta_viewable:.6f}, '
            f'mean |Δviewable_snow_fraction|={mean_abs_delta_viewable:.6f}, '
            f'max |Δraw_snow_fraction|={max_abs_delta_raw_snow:.6f}, '
            f'mean |Δraw_snow_fraction|={mean_abs_delta_raw_snow:.6f}'
        )

    benchmark_scene_output = xr.Dataset(
        data_vars={
            'raw_viewable_snow_fraction_ungrouped': baseline['raw_viewable_snow_fraction'],
            'raw_viewable_snow_fraction_first': per_method['first']['raw_viewable_snow_fraction'],
            'raw_viewable_snow_fraction_chunk_bin_mean': per_method['chunk_bin_mean']['raw_viewable_snow_fraction'],
            'raw_shade_fraction_ungrouped': baseline['raw_shade_fraction'],
            'raw_shade_fraction_first': per_method['first']['raw_shade_fraction'],
            'raw_shade_fraction_chunk_bin_mean': per_method['chunk_bin_mean']['raw_shade_fraction'],
            'raw_canopy_fraction_ungrouped': baseline['raw_canopy_fraction'],
            'raw_canopy_fraction_first': per_method['first']['raw_canopy_fraction'],
            'raw_canopy_fraction_chunk_bin_mean': per_method['chunk_bin_mean']['raw_canopy_fraction'],
            'raw_snow_fraction_ungrouped': baseline['raw_snow_fraction'],
            'raw_snow_fraction_first': per_method['first']['raw_snow_fraction'],
            'raw_snow_fraction_chunk_bin_mean': per_method['chunk_bin_mean']['raw_snow_fraction'],
            'delta_raw_viewable_snow_fraction_first': per_method['first']['raw_viewable_snow_fraction'] - baseline['raw_viewable_snow_fraction'],
            'delta_raw_viewable_snow_fraction_chunk_bin_mean': per_method['chunk_bin_mean']['raw_viewable_snow_fraction'] - baseline['raw_viewable_snow_fraction'],
            'delta_raw_snow_fraction_first': per_method['first']['raw_snow_fraction'] - baseline['raw_snow_fraction'],
            'delta_raw_snow_fraction_chunk_bin_mean': per_method['chunk_bin_mean']['raw_snow_fraction'] - baseline['raw_snow_fraction'],
        },
        coords=baseline['raw_viewable_snow_fraction'].coords,
        attrs={
            'scene': scene_path.name,
            'sensor': benchmark_sensor,
            'tile': benchmark_tile,
            'lut_file': str(benchmark_lut_path),
            'r0_file': str(benchmark_r0_path),
            'first_grouping_tolerance': 0.05,
            'chunk_bin_mean_grouping_tolerance': 0.05,
        },
    )
    benchmark_nc_path = benchmark_output_dir / f"{Path(scene_path).stem}_grouping_benchmark_v2.nc"
    benchmark_scene_output.to_netcdf(benchmark_nc_path)
    benchmark_results.append(
        {
            'scene': scene_path.name,
            'method': 'saved_benchmark_netcdf',
            'seconds': np.nan,
            'minutes': np.nan,
            'grouping_enabled': True,
            'grouping_method': 'all',
            'netcdf': str(benchmark_nc_path),
        }
    )
    print(f'    saved benchmark netCDF: {benchmark_nc_path}')

    del benchmark_scene_output
    del candidate
    del delta_viewable, delta_viewable_values, finite_viewable, flat_delta_viewable
    del delta_raw_snow, delta_raw_snow_values, finite_raw_snow, flat_delta_raw_snow
    gc.collect()

    del baseline
    del per_method
    del prepared_scene
    gc.collect()

benchmark_table = pd.DataFrame(benchmark_results)
benchmark_table


## Grouping Delta Histograms

These histograms show the distribution of per-pixel `raw_snow_fraction` changes for each grouped run relative to the ungrouped baseline.

In [ ]:
histogram_bins = np.linspace(-1.0, 1.0, 81)
delta_variables = ('raw_viewable_snow_fraction', 'raw_snow_fraction')

for scene_name in sorted({scene for scene, _, _ in benchmark_delta_arrays}):
    for delta_name in delta_variables:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
        fig.suptitle(f'NOAA20 grouping deltas vs ungrouped // {scene_name} // {delta_name}', y=1.01)

        for ax, label in zip(axes, ('first', 'chunk_bin_mean')):
            delta = benchmark_delta_arrays.get((scene_name, label, delta_name), np.array([], dtype=np.float64))
            ax.hist(delta, bins=histogram_bins, color='steelblue', edgecolor='white')
            ax.set_title(label)
            ax.set_xlabel(f'Change in {delta_name}')
            ax.set_ylabel('Pixel count')
            ax.axvline(0.0, color='black', linewidth=1.0, linestyle='--')

        fig.tight_layout()
        plt.show()


In [ ]:
benchmark_scene_output

In [ ]:
h[0][-1]